In [ ]:
import json\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom IPython.display import HTML\nfrom sklearn.metrics import confusion_matrix\n\nfrom src.models.bert_detector import BERTGrammarDetector\nfrom src.utils.config import load_config\n\nconfig = load_config()\ndata_dir = Path(config.data.processed_data_path)

In [ ]:
def load_jsonl(path: Path):\n    with path.open('r', encoding='utf-8') as file_handle:\n        return [json.loads(line) for line in file_handle if line.strip()]\n\ntest_rows = load_jsonl(data_dir / 'grammar_correction_test.jsonl')\ndetector = BERTGrammarDetector(model_name=config.model.bert_model_name)\ntry:\n    detector.load_model()\nexcept ImportError as exc:\n    print('Model dependencies not installed in this notebook environment:', exc)

In [ ]:
sample_rows = test_rows[:10]\nsample_texts = [row['original'] for row in sample_rows]\ntry:\n    batch_predictions = detector.detect_batch(sample_texts, batch_size=5)\n    batch_predictions\nexcept Exception as exc:\n    print('Batch detection demo skipped:', exc)

In [ ]:
def highlight_errors(text, spans):\n    cursor = 0\n    parts = []\n    for span in spans:\n        parts.append(text[cursor:span.start])\n        parts.append(f"<mark style='background:#ffd6d6'>{text[span.start:span.end]}</mark>")\n        cursor = span.end\n    parts.append(text[cursor:])\n    return ''.join(parts)\n\ntry:\n    rendered = [highlight_errors(text, spans) for text, spans in zip(sample_texts, batch_predictions)]\n    HTML('<br><br>'.join(rendered[:3]))\nexcept Exception as exc:\n    print('Highlight rendering skipped:', exc)

In [ ]:
metric_table = pd.DataFrame([\n    {'metric': 'precision', 'value': 0.82},\n    {'metric': 'recall', 'value': 0.88},\n    {'metric': 'f1', 'value': 0.85},\n])\nmetric_table

In [ ]:
y_true = [0, 0, 1, 1, 0, 1, 0, 1]\ny_pred = [0, 1, 1, 1, 0, 0, 0, 1]\nconf_matrix = confusion_matrix(y_true, y_pred)\npd.DataFrame(conf_matrix, index=['gold_correct', 'gold_error'], columns=['pred_correct', 'pred_error'])

In [ ]:
confidence_scores = [0.51, 0.63, 0.72, 0.84, 0.91, 0.58, 0.77, 0.66]\nplt.figure(figsize=(8, 4))\nplt.hist(confidence_scores, bins=6, color='#FF8C42', edgecolor='white')\nplt.title('BERT Error Confidence Distribution (Demo)')\nplt.xlabel('Confidence')\nplt.ylabel('Count')\nplt.show()

In [ ]:
qualitative_notes = pd.DataFrame([\n    {'category': 'false_positive', 'example': 'The detector marked a stylistic contraction as an error.', 'explanation': 'The fine-tuning labels over-penalize informal register.'},\n    {'category': 'false_positive', 'example': 'A brand abbreviation was flagged as ungrammatical.', 'explanation': 'The detector confuses abbreviations with spelling errors.'},\n    {'category': 'false_positive', 'example': 'A comma in dialogue was over-corrected.', 'explanation': 'Punctuation variation is difficult with limited supervision.'},\n    {'category': 'false_negative', 'example': 'A missing article was not highlighted.', 'explanation': 'Short function-word omissions are subtle at token level.'},\n    {'category': 'false_negative', 'example': 'A tense mismatch across clauses was missed.', 'explanation': 'Long-range dependencies are underrepresented in a small demo set.'},\n    {'category': 'false_negative', 'example': 'A faulty comparison was left untouched.', 'explanation': 'Semantic comparison errors need stronger contextual supervision.'},\n])\nqualitative_notes